# 01 - Target Definition
**DNA Gene Mapping Project - ML Phase V5 - Feature Selection**  
**Author:** Sharique Mohammad  
**Date:** February 2026

## Purpose
Confirm the target variable for every use case. Verify it exists, check its type,
measure class distribution and imbalance ratio, and flag which use cases need SMOTE
or cross-validation before training begins.

## Output
`data/feature_selection/01_target_definitions.csv`

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

PROJECT_ROOT = Path().absolute().parent.parent
FS_DIR       = PROJECT_ROOT / 'data' / 'feature_selection'
FS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Setup complete")
print(f"Feature selection dir: {FS_DIR}")

## 2. Database Connection

In [ ]:
POSTGRES_HOST     = os.getenv("POSTGRES_HOST")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)
print(f"Connected: {POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")

## 3. Use Case Registry

In [ ]:
USE_CASES = [
    # (uc_code, table, target, target_type, sample_pct, cv_required)
    ('UC01', 'clinical_ml_features',               'target_is_pathogenic',                 'binary',     10,  False),
    ('UC02', 'disease_ml_features',                'is_pathogenic',                        'binary',     10,  False),
    ('UC03', 'pharmacogene_ml_features',           'is_pathogenic',                        'binary',     10,  False),
    ('UC04', 'variant_impact_ml_features',         'is_high_impact',                       'binary',     10,  False),
    ('UC05', 'structural_variant_ml_features',     'sv_classification',                    'multiclass', 100, False),
    ('UC06', 'variant_drug_response_ml_features',  'is_actionable_pharmacogene_variant',   'binary',     10,  False),
    ('UC07', 'variant_cancer_ml_features',         'is_driver_candidate',                  'binary',     10,  False),
    ('UC08', 'variant_population_ml_features',     'is_carrier_screening_candidate',       'binary',     100, False),
    ('UC09', 'population_frequency_ml_features',   'is_clinically_actionable_rare_variant','binary',     100, False),
    ('UC10', 'gene_pharmacogene_ml_features',      'is_high_priority_pharmacogene',        'binary',     100, True),
    ('UC11', 'gene_expression_ml_features',        'is_clinically_relevant_expression',    'binary',     100, False),
    ('UC12', 'gene_protein_family_ml_features',    'is_high_value_protein_family',         'binary',     100, False),
    ('UC13', 'gene_test_availability_ml_features', 'is_high_priority_test_gene',           'binary',     100, False),
    ('UC14', 'transcript_expression_ml_features',  'is_clinically_relevant_expression',    'binary',     100, False),
    ('UC15', 'cancer_variant_ml_features',         'gene_cancer_role',                     'multiclass', 10,  False),
]

def load_sample(table, engine, pct):
    if pct == 100:
        return pd.read_sql(f"SELECT * FROM gold.{table}", engine)
    return pd.read_sql(f"SELECT * FROM gold.{table} TABLESAMPLE SYSTEM ({pct})", engine)

def fix_bool(series):
    return series.astype(str).str.lower().map({'true': True, 'false': False, '1': True, '0': False})

def fix_types(df, schema_df, table):
    tbl = schema_df[schema_df['table_name'] == table]
    for _, row in tbl.iterrows():
        c, dt = row['column_name'], row['data_type']
        if c not in df.columns:
            continue
        if dt in ('INT', 'BIGINT'):
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
        elif dt == 'DOUBLE':
            df[c] = pd.to_numeric(df[c], errors='coerce')
        elif dt == 'BOOLEAN':
            df[c] = fix_bool(df[c])
    return df

print(f"Registry loaded: {len(USE_CASES)} use cases")

## 4. Target Validation

In [ ]:
results = []

for uc, table, target, ttype, pct, cv in USE_CASES:
    print(f"--- {uc} | {table} | target: {target} ---")
    df  = load_sample(table, engine, pct)
    n   = len(df)

    if target not in df.columns:
        print(f"  ERROR: column '{target}' not found"); print(); continue

    if ttype == 'binary':
        vals = fix_bool(df[target])
        pos  = int(vals.sum())
        neg  = n - pos
        ppos = pos / n * 100 if n else 0
        rat  = max(pos, neg) / min(pos, neg) if min(pos, neg) > 0 else 999
        smote = rat > 5
        print(f"  rows={n:,}  pos={pos:,} ({ppos:.1f}%)  neg={neg:,}  ratio={rat:.1f}:1  SMOTE={smote}  CV={cv}")
        results.append({
            'use_case': uc, 'table': table, 'target': target, 'type': ttype,
            'total_rows': n, 'positive': pos, 'negative': neg,
            'positive_pct': round(ppos, 2), 'imbalance_ratio': round(rat, 2),
            'smote_needed': smote, 'cv_required': cv
        })
    else:
        dist = df[target].value_counts()
        rat  = dist.max() / dist.min() if dist.min() > 0 else 999
        print(f"  rows={n:,}  classes={dist.shape[0]}  ratio={rat:.1f}:1")
        for cls, cnt in dist.items():
            print(f"    {str(cls):<35} : {cnt:,} ({cnt/n*100:.1f}%)")
        results.append({
            'use_case': uc, 'table': table, 'target': target, 'type': ttype,
            'total_rows': n, 'positive': int(dist.max()), 'negative': int(dist.min()),
            'positive_pct': round(dist.max()/n*100, 2), 'imbalance_ratio': round(rat, 2),
            'smote_needed': rat > 5, 'cv_required': cv
        })
    print()

## 5. Save Results

In [ ]:
target_df = pd.DataFrame(results)
target_df.to_csv(FS_DIR / '01_target_definitions.csv', index=False)
print("Saved: 01_target_definitions.csv")
print()
print(target_df[['use_case','target','total_rows','positive_pct',
                  'imbalance_ratio','smote_needed','cv_required','type']].to_string(index=False))

## 6. Visualize

In [ ]:
binary_df = target_df[target_df['type'] == 'binary'].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Stacked bar: class split
for i, row in binary_df.iterrows():
    axes[0].barh(i, row['positive_pct'], color='#e74c3c', alpha=0.8)
    axes[0].barh(i, 100 - row['positive_pct'], left=row['positive_pct'], color='#27ae60', alpha=0.8)
    if row['positive_pct'] > 4:
        axes[0].text(row['positive_pct']/2, i, f"{row['positive_pct']:.1f}%",
                     va='center', ha='center', fontsize=8, color='white', fontweight='bold')

axes[0].set_yticks(range(len(binary_df)))
axes[0].set_yticklabels([f"{r['use_case']} {r['target']}" for _, r in binary_df.iterrows()], fontsize=8)
axes[0].set_xlabel('Class split (%)')
axes[0].set_title('Target Class Distribution (red=positive, green=negative)', fontweight='bold')
axes[0].set_xlim(0, 100)
axes[0].grid(axis='x', alpha=0.3)

# Imbalance bar
sorted_df = target_df.sort_values('imbalance_ratio')
colors = ['#e74c3c' if r > 5 else '#3498db' for r in sorted_df['imbalance_ratio']]
bars = axes[1].barh(sorted_df['use_case'], sorted_df['imbalance_ratio'],
                     color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, sorted_df['imbalance_ratio']):
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2.,
                 f'{val:.1f}:1', va='center', fontsize=8, fontweight='bold')
axes[1].axvline(5, color='red', linestyle='--', linewidth=1.5, label='SMOTE threshold')
axes[1].set_xlabel('Imbalance ratio')
axes[1].set_title('Class Imbalance Ratio (red = SMOTE needed)', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(FS_DIR / '01_target_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_overview.png")

## 7. Summary

In [ ]:
smote_list = target_df[target_df['smote_needed']]['use_case'].tolist()
cv_list    = target_df[target_df['cv_required']]['use_case'].tolist()
mc_list    = target_df[target_df['type'] == 'multiclass']['use_case'].tolist()

print("=" * 55)
print("TARGET DEFINITION COMPLETE")
print("=" * 55)
print(f"Total use cases  : {len(results)}")
print(f"Binary           : {(target_df['type']=='binary').sum()}")
print(f"Multiclass       : {(target_df['type']=='multiclass').sum()}")
print(f"SMOTE needed     : {smote_list}")
print(f"CV required      : {cv_list}")
print(f"Multiclass UCs   : {mc_list}")
print()
print("Next: 02_feature_quality_checks.ipynb")